In [3]:
# To prevent Colab from disconnecting: https://sjkoding.tistory.com/79

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# upload data.zip
from google.colab import files
files.upload()

In [ ]:
!unzip -q data.zip

In [5]:
cd data

/home/heayoun/heayoun/machine_learning/data


In [6]:
import os
import re
import time
import random
import pickle
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import nltk
from PIL import Image

import torch
import torch.nn as nn
import torch.utils.data as data
import torchvision.models as models

from torchvision import transforms
from torch.nn.utils.rnn import pack_padded_sequence
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score


In [7]:
# =========================================================
# 1. Config / setup
# =========================================================

# ============================== !!!
# DO NOT CHANGE THIS STRING.
# This is tied to dataset labeling / unknown-caption handling.
# Changing this can break both training/evaluation consistency.
# ============================== !!!
UNKNOWN_CAPTION = "I don't know what is in this image."

# Metadata file used to recover unknown labels from the dataset.
META_PATH = "./dataset/caption_metadata.pkl"


@dataclass
class Config:
    seed: int = 1
    crop_size: int = 224
    batch_size: int = 128
    num_workers: int = 2

    # ============================== !!!
    # DO NOT CHANGE ANY OF THE FIELDS BELOW.
    #
    # These fields directly affect model tensor shapes and checkpoint compatibility:
    # - embed_size
    # - hidden_size
    # - num_layers
    #
    # If you change any of them, evaluation will fail.
    # ============================== !!!
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 1

    num_epochs: int = 10
    learning_rate: float = 1e-3

    # ============================== !!!
    # DO NOT CHANGE THIS.
    # It changes evaluation behavior.
    # ============================== !!!
    unknown_threshold: float = 0.7

    train_image_dir: str = "./dataset/train_images"
    valid_image_dir: str = "./dataset/train_images"
    train_captions_path: str = "./dataset/train_captions.pkl"
    valid_captions_path: str = "./dataset/train_captions.pkl"

    # ============================== !!!
    # DO NOT CHANGE THIS VOCAB FILE.
    # The saved checkpoints assume the exact same vocabulary mapping:
    # token <-> index
    # ============================== !!!
    vocab_path: str = "./vocab.pkl"

    model_dir: str = "./models"

    encoder_ckpt_name: str = "encoder-best.ckpt"
    decoder_ckpt_name: str = "decoder-best.ckpt"


cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=1):
    """Seed everything for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def download_nltk_resources():
    """Download NLTK resources used by tokenization and METEOR."""
    for name in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
        try:
            nltk.download(name, quiet=True)
        except Exception:
            pass


def ensure_dir(path):
    """Create the directory if it does not exist."""
    os.makedirs(path, exist_ok=True)


def unwrap_model(model):
    """Return the original module if DataParallel is used."""
    return model


def save_model_state(model, path):
    """Save only the state_dict of the unwrapped model."""
    torch.save(unwrap_model(model).state_dict(), path)


# ============================== !!!
# DO NOT CHANGE THIS LOADING CONTRACT.
# Evaluation depends on loading raw state_dict into the exact same module structure.
# ============================== !!!
def load_model_state(model, path, map_location=device):
    state = torch.load(path, map_location=map_location)
    unwrap_model(model).load_state_dict(state)


seed_everything(cfg.seed)
download_nltk_resources()
ensure_dir(cfg.model_dir)

In [15]:
# =========================================================
# 2. Text preprocessing
# =========================================================
# ============================== !!!
# DO NOT CHANGE THESE FUNCTIONS.
# train/eval must share the exact same text preprocessing rules:
# - clean_caption
# - tokenize_caption
# - encode/decode assumptions
# If you modify these, BLEU/METEOR comparison and vocab behavior can break.
# ============================== !!!

def clean_caption(caption):
    """Normalize a caption string and filter out unusable examples."""
    caption = str(caption).strip().replace("\n", " ").replace("\t", " ")
    caption = " ".join(caption.split())
    if not caption:
        return None

    if caption == UNKNOWN_CAPTION:
        return caption

    lowered = caption.lower()
    junk_markers = [
        "strange caption:",
        "caption:",
        "rewrite the caption",
        "do not include",
        "keep it short",
    ]
    if any(marker in lowered for marker in junk_markers):
        return None

    letter_ratio = sum(ch.isalpha() for ch in caption) / max(1, len(caption))
    if letter_ratio < 0.55:
        return None

    tokens = re.findall(r"[a-zA-Z0-9']+", lowered)
    if not 2 <= len(tokens) <= 25:
        return None

    return lowered


def tokenize_caption(text, max_len=20):
    """Tokenize caption text using the same rule for train and eval."""
    tokens = nltk.tokenize.word_tokenize(str(text).lower())
    tokens = [re.sub(r"[^a-z0-9']+", "", token) for token in tokens]
    tokens = [token for token in tokens if token]
    return tokens[:max_len]


def encode_caption(caption, vocab, max_len=20):
    """Convert a caption string into token ids."""
    tokens = tokenize_caption(caption, max_len=max_len)
    ids = [vocab("<start>")] + [vocab(token) for token in tokens] + [vocab("<end>")]
    return torch.tensor(ids, dtype=torch.long)


def idx_to_word(vocab, idx):
    """Map an index back to a vocabulary token."""
    return getattr(vocab, "idx2word", {}).get(int(idx), "")


def decode_ids(ids, vocab, stop_at_end=True):
    """Convert predicted token ids back into words."""
    words = []
    for token_id in ids:
        word = idx_to_word(vocab, int(token_id))
        if word in {"<start>", "<pad>"}:
            continue
        if stop_at_end and word == "<end>":
            break
        if not word:
            continue
        words.append(word)
    return words

In [16]:
# =========================================================
# 3. Dataset / dataloader
# =========================================================

# ============================== !!!
# DO NOT CHANGE THE VOCABULARY INTERFACE OR TOKEN MAPPING RULES.
#
# Evaluation assumes that:
# - the same vocab.pkl is used
# - the same token-to-index mapping is preserved
# - special tokens such as <start>, <end>, <unk>, <pad> behave the same way
# ============================== !!!
class Vocabulary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.idx = 0

    def add_word(self, word):
        if word not in self.word2idx:
            self.word2idx[word] = self.idx
            self.idx2word[self.idx] = word
            self.idx += 1

    def __call__(self, word):
        if word not in self.word2idx:
            if "" in self.word2idx:
                return self.word2idx[""]
            if "<unk>" in self.word2idx:
                return self.word2idx["<unk>"]
            raise KeyError(f"Unknown token: {word}")
        return self.word2idx[word]

    def __len__(self):
        return len(self.word2idx)


class ImageCaptioningDataset(data.Dataset):
    def __init__(
        self,
        root,
        captions_path,
        vocab,
        transform=None,
        is_train=True,
        meta_path=None,
        max_caption_len=50,
    ):
        self.root = root
        self.vocab = vocab
        self.transform = transform
        self.is_train = is_train
        self.max_caption_len = max_caption_len
        self.samples = []

        with open(captions_path, "rb") as f:
            captions = pickle.load(f)

        meta_by_fname = {}
        if meta_path and os.path.exists(meta_path):
            with open(meta_path, "rb") as f:
                meta_rows = pickle.load(f)

            split_name = "train" if is_train else "test"
            meta_by_fname = {
                row["fname"]: row
                for row in meta_rows
                if row["split"] == split_name
            }

        for fname, caps in captions:
            row = meta_by_fname.get(fname, {})
            is_unknown = bool(row.get("is_unknown_cluster", False))

            # ============================== !!!
            # DO NOT MODIFY THIS CLEANING CALL
            #
            # clean_caption() defines the canonical text preprocessing
            # shared by BOTH training and evaluation.
            # ============================== !!!
            cleaned_caps = [clean_caption(cap) for cap in caps]

            # ============================== !!!
            # SAFE TO MODIFY (TRAINING ONLY)
            #
            # You may modify filtering behavior to experiment
            # with noisy training data.
            # ============================== !!!
            cleaned_caps = [cap for cap in cleaned_caps if cap is not None]
            if not cleaned_caps:
                continue

            if is_unknown or all(cap == UNKNOWN_CAPTION for cap in cleaned_caps):
                self.samples.append(
                    {"fname": fname, "caption": UNKNOWN_CAPTION, "is_unknown": 1}
                )
                continue

            base_caps = [cap for cap in cleaned_caps if cap != UNKNOWN_CAPTION]
            self.samples.extend(
                {"fname": fname, "caption": cap, "is_unknown": 0}
                for cap in base_caps
            )

        # Print dataset statistics only for the training split.
        if is_train:
            unique_images = len({item["fname"] for item in self.samples})
            print(
                f"[Train Dataset] samples={len(self.samples)}, unique_images={unique_images}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        item = self.samples[index]
        image = Image.open(os.path.join(self.root, item["fname"])).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        # ============================== !!!
        # DO NOT MODIFY encode_caption CALL
        #
        # encode_caption defines the tokenization and
        # vocabulary mapping used by the model.
        # ============================== !!!
        target = encode_caption(item["caption"], self.vocab, max_len=self.max_caption_len)
        return image, target, item["is_unknown"]


def collate_batch(batch, sort_by_length=True):
    """Pad captions and stack a mini-batch."""
    if sort_by_length:
        batch = sorted(batch, key=lambda x: len(x[1]), reverse=True)

    images, captions, is_unknowns = zip(*batch)
    images = torch.stack(images, dim=0)

    lengths = [len(caption) for caption in captions]
    targets = torch.zeros(len(captions), max(lengths), dtype=torch.long)

    for i, caption in enumerate(captions):
        targets[i, :lengths[i]] = caption[:lengths[i]]

    is_unknowns = torch.tensor(is_unknowns, dtype=torch.float32)
    return images, targets, lengths, is_unknowns


def build_transforms(crop_size=224):
    """Build train/eval image transforms."""
    train_transform = transforms.Compose([
        transforms.RandomCrop(crop_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])

    # ============================== !!!
    # DO NOT CHANGE THIS EVAL TRANSFORM LIGHTLY
    # ============================== !!!
    valid_transform = transforms.Compose([
        transforms.Resize(crop_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    return train_transform, valid_transform


def get_loader(
    root,
    captions_path,
    vocab,
    transform,
    batch_size,
    shuffle,
    num_workers,
    is_train,
):
    """Create a dataloader for captioning."""
    dataset = ImageCaptioningDataset(
        root=root,
        captions_path=captions_path,
        vocab=vocab,
        transform=transform,
        is_train=is_train,
        meta_path=META_PATH,
    )

    return torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_batch,
    )

In [17]:
# =========================================================
# 4. Model
# =========================================================
# ============================== !!!
# DO NOT CHANGE THE MODEL STRUCTURE OR ANY SHAPE-RELATED CONFIG
# IF YOU WANT TO LOAD EXISTING CHECKPOINTS.
#
# This includes:
# - embed_size
# - hidden_size
# - num_layers
# - backbone choice (e.g. resnet50)
# - module names
# - tensor shapes
#
# The following modules must remain structurally compatible with saved state_dicts:
# - EncoderCNN.resnet
# - EncoderCNN.linear
# - EncoderCNN.bn
# - EncoderCNN.unknown_head
# - DecoderRNN.embed
# - DecoderRNN.lstm
# - DecoderRNN.linear
#
# If you rename/remove/change shapes, checkpoint loading will break.
# ============================== !!!

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super().__init__()

        try:
            weights = models.ResNet50_Weights.IMAGENET1K_V1
            resnet = models.resnet50(weights=weights)
        except Exception:
            resnet = models.resnet50(pretrained=True)

        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

        # ============================== !!!
        # DO NOT REMOVE THIS MODULE.
        # ============================== !!!
        self.unknown_head = nn.Sequential(
            nn.Linear(embed_size, embed_size),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(embed_size, 1),
        )

        self.backbone_trainable = any(param.requires_grad for param in self.resnet.parameters())

    def forward(self, images, return_unknown_logit=False):
        if self.backbone_trainable:
            features = self.resnet(images)
        else:
            with torch.no_grad():
                features = self.resnet(images)

        features = features.reshape(features.size(0), -1)
        features = self.bn(self.linear(features))

        if return_unknown_logit:
            unknown_logit = self.unknown_head(features).squeeze(1)
            return features, unknown_logit
        return features


class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers, max_seq_length=20):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.max_seq_length = max_seq_length

    def forward(self, features, captions, lengths):
        embeddings = self.embed(captions)
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)
        packed = pack_padded_sequence(
            embeddings, lengths, batch_first=True, enforce_sorted=True
        )
        hiddens, _ = self.lstm(packed)
        return self.linear(hiddens[0])

    def sample(self, features, states=None):
        sampled_ids = []
        inputs = features.unsqueeze(1)

        for _ in range(self.max_seq_length):
            hiddens, states = self.lstm(inputs, states)
            outputs = self.linear(hiddens.squeeze(1))
            predicted = outputs.argmax(dim=1)
            sampled_ids.append(predicted)
            inputs = self.embed(predicted).unsqueeze(1)

        return torch.stack(sampled_ids, dim=1)

In [11]:
# =========================================================
# 5. Evaluation helpers
# =========================================================
# ============================== !!!
# DO NOT CHANGE THESE.
# ============================== !!!

def save_captioned_image(img_path, caption_text, save_path, text_height=80):
    """Save an image with the generated caption rendered below it."""
    from PIL import ImageDraw, ImageFont

    def text_width(draw, text, font):
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0]

    def wrap_text_to_width(draw, text, font, max_width):
        words = str(text).split()
        lines = []
        current = ""

        for word in words:
            candidate = word if current == "" else current + " " + word

            if text_width(draw, candidate, font) <= max_width:
                current = candidate
            else:
                if current:
                    lines.append(current)

                if text_width(draw, word, font) <= max_width:
                    current = word
                else:
                    chunk = ""
                    for ch in word:
                        candidate_chunk = chunk + ch
                        if text_width(draw, candidate_chunk, font) <= max_width:
                            chunk = candidate_chunk
                        else:
                            if chunk:
                                lines.append(chunk)
                            chunk = ch
                    current = chunk

        if current:
            lines.append(current)

        return lines if lines else [""]

    img_vis = Image.open(img_path).convert("RGB")
    w, h = img_vis.size

    font = ImageFont.load_default()

    temp_img = Image.new("RGB", (w, 100), (255, 255, 255))
    temp_draw = ImageDraw.Draw(temp_img)

    margin = 10
    max_text_width = max(1, w - 2 * margin)
    lines = wrap_text_to_width(temp_draw, caption_text, font, max_text_width)

    line_heights = []
    for line in lines:
        bbox = temp_draw.textbbox((0, 0), line, font=font)
        line_heights.append(bbox[3] - bbox[1])

    line_spacing = 4
    total_text_height = sum(line_heights) + line_spacing * max(0, len(lines) - 1)

    text_height = max(text_height, total_text_height + 2 * margin)

    new_img = Image.new("RGB", (w, h + text_height), (255, 255, 255))
    new_img.paste(img_vis, (0, 0))

    draw = ImageDraw.Draw(new_img)

    y = h + (text_height - total_text_height) // 2

    for line, line_h in zip(lines, line_heights):
        line_w = text_width(draw, line, font)
        x = max(margin, (w - line_w) // 2)
        draw.text((x, y), line, fill=(0, 0, 0), font=font)
        y += line_h + line_spacing

    new_img.save(save_path)


def build_references_from_pkl(captions_pkl_path):
    """Build tokenized reference captions from a pickle file."""
    with open(captions_pkl_path, "rb") as f:
        items = pickle.load(f)

    references = defaultdict(list)
    for fname, captions in items:
        for caption in captions:
            cleaned = clean_caption(caption)
            if cleaned is None:
                continue
            tokens = tokenize_caption(cleaned)
            if tokens:
                references[fname].append(tokens)

    return {fname: refs for fname, refs in references.items() if refs}


@torch.no_grad()
def generate_caption_for_image(
    encoder,
    decoder,
    image_tensor,
    vocab,
    device,
    unknown_threshold=0.5,
):
    """Generate a caption for a single image."""
    encoder.eval()
    decoder.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)
    encoder_model = unwrap_model(encoder)
    decoder_model = unwrap_model(decoder)

    # ============================== !!!
    # DO NOT REMOVE THIS CALL STYLE.
    # ============================== !!!
    features, _ = encoder_model(image_tensor, return_unknown_logit=True)

    sampled = decoder_model.sample(features)
    return decode_ids(sampled[0].cpu().tolist(), vocab, stop_at_end=True)


def compute_bleu4_meteor(hypotheses, references):
    """Compute BLEU-4 and METEOR only."""
    smoothie = SmoothingFunction().method4
    keys = sorted(set(hypotheses) & set(references))

    list_of_references = [references[key] for key in keys]
    predicted_tokens = [hypotheses[key] for key in keys]

    bleu4 = float(
        corpus_bleu(
            list_of_references,
            predicted_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothie,
        )
    )

    meteor_scores = []
    for key in keys:
        try:
            meteor_scores.append(meteor_score(references[key], hypotheses[key]))
        except Exception:
            pass

    meteor = float(np.mean(meteor_scores)) if meteor_scores else 0.0
    return {
        "BLEU-4": bleu4,
        "METEOR": meteor,
        "N_eval": len(keys),
    }


def to_scaled_100(raw_value, max_value):
    """Convert a raw metric into a 100-point based score."""
    return (raw_value / max_value) * 100.0 if max_value > 0 else 0.0

def print_metric_report(metrics):
    """Print BLEU-4 / METEOR as raw values and 100-point scaled values."""
    bleu4 = metrics["BLEU-4"]
    meteor = metrics["METEOR"]

    bleu4_score = to_scaled_100(bleu4, 0.20)
    meteor_score_100 = to_scaled_100(meteor, 0.30)
    
    if bleu4_score > 0 and meteor_score_100 > 0:
        harmonic_score = 2 / (1 / bleu4_score + 1 / meteor_score_100)
    else:
        harmonic_score = 0.0

    print("\n[ Evaluation Report ]")
    print(f"N_eval         : {metrics['N_eval']}")
    print(f"BLEU-4 raw     : {bleu4:.4f}")
    print(f"BLEU-4 score   : {bleu4_score:.2f} points  (100 pts = BLEU-4 0.20)")
    print(f"METEOR raw     : {meteor:.4f}")
    print(f"METEOR score   : {meteor_score_100:.2f} points  (100 pts = METEOR 0.30)")
    print(f"HARMONIC score : {harmonic_score:.2f} points")


@torch.no_grad()
def evaluate_captioning_metrics(
    encoder,
    decoder,
    image_dir,
    captions_pkl_path,
    transform,
    vocab,
    device,
    unknown_threshold=0.5,
):
    """Run evaluation on all reference images."""
    save_dir = "./eval_outputs"
    os.makedirs(save_dir, exist_ok=True)

    save_count = 0
    max_save = 10

    references = build_references_from_pkl(captions_pkl_path)

    hypotheses = {}
    fnames = sorted(references.keys())

    for fname in fnames:
        img_path = os.path.join(image_dir, fname)
        if not os.path.exists(img_path):
            continue

        image = Image.open(img_path).convert("RGB")
        image = transform(image) if transform is not None else image

        pred_tokens = generate_caption_for_image(
            encoder=encoder,
            decoder=decoder,
            image_tensor=image,
            vocab=vocab,
            device=device,
            unknown_threshold=unknown_threshold,
        ) or [""]

        hypotheses[fname] = pred_tokens

        if save_count < max_save:
            caption_text = " ".join(pred_tokens)
            save_path = os.path.join(save_dir, fname)
            save_captioned_image(img_path, caption_text, save_path)
            save_count += 1

    print(f"\nSaved {save_count} visualization images to: {os.path.abspath(save_dir)}")

    return compute_bleu4_meteor(hypotheses, references)

In [12]:
# =========================================================
# 6. TRAIN
# =========================================================
# Minimal training code.
# Do not touch evaluation-critical code.

def build_optimizer(encoder, decoder, learning_rate):
    """Build a minimal optimizer for training."""
    encoder_core = unwrap_model(encoder)

    params = (
        list(encoder_core.linear.parameters())
        + list(encoder_core.bn.parameters())
        + list(decoder.parameters())
    )
    return torch.optim.Adam(params, lr=learning_rate)


def compute_caption_loss(decoder, features, captions, lengths, criterion):
    """Compute caption cross-entropy loss."""
    targets = pack_padded_sequence(
        captions,
        lengths,
        batch_first=True,
        enforce_sorted=True,
    )[0]
    outputs = decoder(features, captions, lengths)
    return criterion(outputs, targets)


def run_train_epoch(encoder, decoder, data_loader, criterion, optimizer, device, log_step=20):
    """Run one training epoch."""
    encoder.train()
    decoder.train()

    total_loss = 0.0
    total_samples = 0

    for step, (images, captions, lengths, _) in enumerate(data_loader):
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        features = encoder(images)
        loss = compute_caption_loss(decoder, features, captions, lengths, criterion)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_samples += images.size(0)

        if step % log_step == 0:
            print(f"Step [{step}/{len(data_loader)}] Loss: {loss.item():.4f}")

    avg_loss = total_loss / max(1, total_samples)
    return {
        "avg_loss": avg_loss,
        "perplexity": float(np.exp(avg_loss)),
    }


def train_main():
    """Train the model and save checkpoints."""
    with open(cfg.vocab_path, "rb") as f:
        vocab = pickle.load(f)

    train_transform, _ = build_transforms(cfg.crop_size)

    train_loader = get_loader(
        root=cfg.train_image_dir,
        captions_path=cfg.train_captions_path,
        vocab=vocab,
        transform=train_transform,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        is_train=True,
    )

    encoder = EncoderCNN(embed_size=cfg.embed_size).to(device)
    decoder = DecoderRNN(
        embed_size=cfg.embed_size,
        hidden_size=cfg.hidden_size,
        vocab_size=len(vocab),
        num_layers=cfg.num_layers,
    ).to(device)

    pad_idx = vocab("<pad>") if "<pad>" in getattr(vocab, "word2idx", {}) else 0
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = build_optimizer(encoder, decoder, cfg.learning_rate)

    best_ppl = float("inf")

    for epoch in range(cfg.num_epochs):
        print(f"\n========== TRAIN Epoch {epoch + 1}/{cfg.num_epochs} ==========")
        metrics = run_train_epoch(
            encoder=encoder,
            decoder=decoder,
            data_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )

        print(
            f"[Train] epoch={epoch + 1}, "
            f"avg_loss={metrics['avg_loss']:.4f}, "
            f"perplexity={metrics['perplexity']:.4f}"
        )

        encoder_ckpt_path = os.path.join(cfg.model_dir, f"encoder-{epoch + 1}.ckpt")
        decoder_ckpt_path = os.path.join(cfg.model_dir, f"decoder-{epoch + 1}.ckpt")
        save_model_state(encoder, encoder_ckpt_path)
        save_model_state(decoder, decoder_ckpt_path)

        if metrics["perplexity"] <= best_ppl:
            best_ppl = metrics["perplexity"]
            save_model_state(encoder, os.path.join(cfg.model_dir, cfg.encoder_ckpt_name))
            save_model_state(decoder, os.path.join(cfg.model_dir, cfg.decoder_ckpt_name))
            print("[Saved best checkpoints]")

In [13]:
# =========================================================
# 7. EVAL
# =========================================================
# ============================== !!!
# DO NOT CHANGE THIS ENTRY LOGIC.
# This block is the intended evaluation path:
# 1) build model with the exact same structure
# 2) load saved checkpoints
# 3) run evaluation helpers
# ============================== !!!

def eval_main(
    encoder_ckpt_path=None,
    decoder_ckpt_path=None,
):
    """Load checkpoints and run evaluation."""
    if encoder_ckpt_path is None:
        encoder_ckpt_path = os.path.join(cfg.model_dir, cfg.encoder_ckpt_name)
    if decoder_ckpt_path is None:
        decoder_ckpt_path = os.path.join(cfg.model_dir, cfg.decoder_ckpt_name)

    with open(cfg.vocab_path, "rb") as f:
        vocab = pickle.load(f)

    _, valid_transform = build_transforms(cfg.crop_size)

    encoder = EncoderCNN(embed_size=cfg.embed_size).to(device)
    decoder = DecoderRNN(
        embed_size=cfg.embed_size,
        hidden_size=cfg.hidden_size,
        vocab_size=len(vocab),
        num_layers=cfg.num_layers,
    ).to(device)

    if not os.path.exists(encoder_ckpt_path):
        raise FileNotFoundError(f"Encoder checkpoint not found: {encoder_ckpt_path}")
    if not os.path.exists(decoder_ckpt_path):
        raise FileNotFoundError(f"Decoder checkpoint not found: {decoder_ckpt_path}")

    load_model_state(encoder, encoder_ckpt_path, map_location=device)
    load_model_state(decoder, decoder_ckpt_path, map_location=device)

    print(f"[Loaded] {encoder_ckpt_path}")
    print(f"[Loaded] {decoder_ckpt_path}")

    metrics = evaluate_captioning_metrics(
        encoder=encoder,
        decoder=decoder,
        image_dir=cfg.valid_image_dir,
        captions_pkl_path=cfg.valid_captions_path,
        transform=valid_transform,
        vocab=vocab,
        device=device,
        unknown_threshold=cfg.unknown_threshold,
    )

    print_metric_report(metrics)

In [ ]:
# =========================================================
# 8. Entry point
# =========================================================

if __name__ == "__main__":
    MODE = "train"   # "train" or "eval"

    if MODE == "train":
        train_main()
    elif MODE == "eval":
        eval_main()
    else:
        raise ValueError("MODE must be either 'train' or 'eval'")